In [ ]:
import pandas as pd

df = pd.read_csv("/Users/ziadsaad/Desktop/225 new project/eco225-project-zeyadsaad1/uhaul_scrape/uhaul_migration_2010_2023.csv")

print("HEAD:")
print(df.head(), "\n")

print("TAIL:")
print(df.tail(), "\n")

print("SHAPE:")
print(df.shape, "\n")

print("COLUMNS:")
print(df.columns.tolist(), "\n")

print("YEARS PRESENT:")
print(sorted(df["year"].unique()), "\n")

print("STATES PER YEAR:")
print(df.groupby("year").size(), "\n")

dup_count = df.duplicated(["year", "state_abbr"]).sum()
print("DUPLICATES BY (year, state_abbr):", dup_count, "\n")

print("MISSING VALUES:")
print(df.isna().sum(), "\n")

print("SUMMARY STATS:")
print(df[["growth_rank", "in_migration_pct", "out_migration_pct", "net_migration_pct"]].describe(), "\n")

print("ROWS WITH MISSING KEY VALUES:")
missing_key = df[df[["growth_rank", "in_migration_pct", "out_migration_pct"]].isna().any(axis=1)]
print(missing_key.head(20), "\n")
print("Total rows with missing key values:", len(missing_key), "\n")

df = df.sort_values(["state_abbr", "year"]).reset_index(drop=True)

df.to_csv("uhaul_migration_clean.csv", index=False)
print("Saved cleaned dataset as: uhaul_migration_clean.csv")

HEAD:
   year       state state_abbr  growth_rank  in_migration_pct  \
0  2015     Alabama         AL         11.0              50.7   
1  2015      Alaska         AK         31.0              50.0   
2  2015     Arizona         AZ         15.0              50.3   
3  2015    Arkansas         AR          8.0              51.2   
4  2015  California         CA          5.0              50.6   

   out_migration_pct                                   leading_cities  \
0               49.3                                              NaN   
1               50.0                                              NaN   
2               49.7                                              NaN   
3               48.8                                              NaN   
4               49.4  #1 Concord, #2 Sacramento-Roseville, #9 Manteca   

   net_migration_pct  
0                1.4  
1                0.0  
2                0.6  
3                2.4  
4                1.2   

TAIL:
     year         

In [ ]:
import pandas as pd

df = pd.read_csv("uhaul_migration_clean.csv")

print("Original shape:", df.shape)

df = df[df["state_abbr"] != "HI"]

df = df.drop(columns=["leading_cities"])

df = df.sort_values(["state_abbr","year"]).reset_index(drop=True)

print("\nAfter cleaning:", df.shape)

print("\nMissing values:")
print(df.isna().sum())

df.to_csv("uhaul_migration_final.csv", index=False)

print("\nSaved: uhaul_migration_final.csv")

Original shape: (450, 8)

After cleaning: (441, 7)

Missing values:
year                 0
state                0
state_abbr           0
growth_rank          0
in_migration_pct     0
out_migration_pct    0
net_migration_pct    0
dtype: int64

Saved: uhaul_migration_final.csv


In [ ]:
import pandas as pd

main_path = "/Users/ziadsaad/Desktop/225 new project/eco225-project-zeyadsaad1/notebooks/output/final_state_dataset.csv"
uhaul_path = "/Users/ziadsaad/Desktop/225 new project/eco225-project-zeyadsaad1/uhaul_scrape/uhaul_migration_final.csv"

main_df = pd.read_csv(main_path)
uhaul_df = pd.read_csv(uhaul_path)

print("MAIN DATASET")
print(main_df.head())
print(main_df.shape)
print(main_df.columns.tolist(), "\n")

print("UHAUL DATASET")
print(uhaul_df.head())
print(uhaul_df.shape)
print(uhaul_df.columns.tolist(), "\n")

print("Does main dataset have a year column?", "year" in main_df.columns)

if "state_abbr" in main_df.columns:
    print("\nRows per state in main dataset:")
    print(main_df["state_abbr"].value_counts().head())

print("\nDuplicate state-year rows in U-Haul:",
      uhaul_df.duplicated(["state_abbr", "year"]).sum())

MAIN DATASET
        state state_abbr  listings      avg_price  med_price    avg_ppsf  \
0      Alaska         AK       967  491531.053775   424900.0  240.219135   
1     Alabama         AL     21418  353789.048090   279900.0  160.643367   
2    Arkansas         AR     12736  328525.735945   250000.0  151.520380   
3     Arizona         AZ     57550  533331.442780   420000.0  263.297200   
4  California         CA    196580  972748.312717   699999.0  520.207223   

     med_ppsf  avg_house_size  med_house_size  share_ready_to_build  \
0  223.631579     2204.154085          1858.0              0.000000   
1  142.310965     2147.718508          1897.0              0.037445   
2  139.227660     2096.529130          1800.0              0.000550   
3  245.006916     1925.102276          1736.0              0.016629   
4  444.680109     1883.218211          1618.0              0.005738   

   unemp_volatility  unemp_mean  unemp_min  unemp_max  hpi_growth_2010_2023  \
0          1.341223    6

In [ ]:
import pandas as pd

main_path = "/Users/ziadsaad/Desktop/225 new project/eco225-project-zeyadsaad1/notebooks/output/final_state_dataset.csv"
uhaul_path = "/Users/ziadsaad/Desktop/225 new project/eco225-project-zeyadsaad1/uhaul_scrape/uhaul_migration_final.csv"

main_df = pd.read_csv(main_path)
uhaul_df = pd.read_csv(uhaul_path)

migration_state = (
    uhaul_df
    .groupby("state_abbr", as_index=False)
    .agg(
        net_migration_mean=("net_migration_pct", "mean"),
        net_migration_vol=("net_migration_pct", "std"),
    )
)

print("Migration summary preview:")
print(migration_state.head())

final_df = main_df.merge(migration_state, on="state_abbr", how="left")

print("\nFinal shape:", final_df.shape)
print("\nMissing values after merge:")
print(final_df[["net_migration_mean", "net_migration_vol"]].isna().sum())

final_df.to_csv("final_state_with_migration.csv", index=False)

print("\nSaved: final_state_with_migration.csv")

Migration summary preview:
  state_abbr  net_migration_mean  net_migration_vol
0         AK           -0.288889           2.172812
1         AL            0.288889           0.664162
2         AR            0.688889           1.221111
3         AZ            0.200000           0.670820
4         CA           -0.555556           0.785988

Final shape: (50, 21)

Missing values after merge:
net_migration_mean    1
net_migration_vol     1
dtype: int64

Saved: final_state_with_migration.csv


In [ ]:
import pandas as pd

df = pd.read_csv("final_state_with_migration.csv")

print("Before:", df.shape)

df = df.dropna(subset=["net_migration_mean","net_migration_vol"])

print("After:", df.shape)

print("\nRemaining missing:")
print(df[["net_migration_mean","net_migration_vol"]].isna().sum())

df.to_csv("final_state_with_migration_clean.csv", index=False)

print("\nSaved: final_state_with_migration_clean.csv")

Before: (50, 21)
After: (49, 21)

Remaining missing:
net_migration_mean    0
net_migration_vol     0
dtype: int64

Saved: final_state_with_migration_clean.csv
